# COMP5318 Assignment 1: Rice Classification

##### Group number: 54
##### Student 1 SID: 550344487
##### Student 2 SID: 550894775
##### Student 3 SID: ... 
##### Student 4 SID: ... 

## **1. Data Pre-processing**

In [22]:
# Import all libraries
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.impute import SimpleImputer # missing value
from sklearn.preprocessing import MinMaxScaler # normalisation
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
import pandas as pd
import numpy as np

In [23]:
# Ignore future warnings
from warnings import simplefilter
simplefilter(action='ignore', category=FutureWarning)

In [24]:
# Load the rice dataset: rice-final2.csv
df = pd.read_csv('rice-final2.csv')
col = df.columns
class_col = col[-1]
df.head(5)


,Area,Perimiter,Major_Axis_Length,Minor_Axis_Length,Eccentricity,Convex_Area,Extent,class
0,12573,461.4660034,192.9033508,84.57207489,0.898771763,12893,0.550433397,class2
1,12845,464.1210022,194.3322144,85.52433777,0.897951961,13125,0.774962306,class2
2,14055,488.7489929,207.7517548,87.25032806,0.907536149,14484,0.550076306,class1
3,14412,490.3240051,207.4761353,89.68951416,0.901735425,14703,0.598853171,class1
4,14658,477.1170044,189.5666351,99.99777985,0.849550545,15048,0.649503708,class2


In [25]:
# check how many missing value for each col
col = df.columns
class_col = col[-1]

for i in col:
    n_missing = df[df[i] == "?"][i].count()
    print(f'Missing value count in {i}: {n_missing}')

Missing value count in Area: 4
Missing value count in Perimiter: 4
Missing value count in Major_Axis_Length: 5
Missing value count in Minor_Axis_Length: 3
Missing value count in Eccentricity: 6
Missing value count in Convex_Area: 5
Missing value count in Extent: 2
Missing value count in class: 0


In [26]:
# Pre-process dataset
# replace '?' with column mean using sklearn.impute.SimpleImputer
exe_df = df.copy()
imputer = SimpleImputer(strategy='mean')
col_class = exe_df[class_col]
# remove class column to prevent error
exe_df = exe_df.drop(columns=class_col)
exe_df = exe_df.replace('?', np.nan).astype(float)
imp_df = imputer.fit_transform(exe_df)  # return array
imp_df = pd.DataFrame(imp_df, columns=exe_df.columns, index=exe_df.index)
imp_df[class_col] = col_class

In [27]:
# check if any '?' in the data
col = imp_df.columns

for i in col:
    n_missing = imp_df[imp_df[i] == "?"][i].count()
    print(f'Missing value count in {i}: {n_missing}')

Missing value count in Area: 0
Missing value count in Perimiter: 0
Missing value count in Major_Axis_Length: 0
Missing value count in Minor_Axis_Length: 0
Missing value count in Eccentricity: 0
Missing value count in Convex_Area: 0
Missing value count in Extent: 0
Missing value count in class: 0


In [28]:
# normalisation
nor_df = imp_df.copy()
scaler = MinMaxScaler()
col_class = nor_df[class_col]
nor_df = nor_df.drop(columns=class_col)
nored_df = scaler.fit_transform(nor_df)
nored_df = pd.DataFrame(nored_df, columns=nor_df.columns, index=nor_df.index)
for i in nored_df.columns:
    print(f'{i} - min_value: {nored_df[i].min()}, max_value: {nored_df[i].max()}')
nored_df[class_col] = col_class

Area - min_value: 0.0, max_value: 1.0
Perimiter - min_value: 0.0, max_value: 0.9999999999999998
Major_Axis_Length - min_value: 0.0, max_value: 1.0000000000000002
Minor_Axis_Length - min_value: 0.0, max_value: 1.0
Eccentricity - min_value: 0.0, max_value: 1.0
Convex_Area - min_value: 0.0, max_value: 1.0
Extent - min_value: 0.0, max_value: 1.0


In [29]:
# replace class
nored_df[class_col] = nored_df[class_col].map({'class1': 0, 'class2': 1})
nored_df[class_col] = nored_df[class_col].astype(int)
processed_df = nored_df.copy()

In [30]:
# Print first ten rows of pre-processed dataset to 4 decimal places as per assignment spec
# A function is provided to assist

def print_data(X, y, n_rows=10):
    """Takes a numpy data array and target and prints the first ten rows.
    
    Arguments:
        X: numpy array of shape (n_examples, n_features)
        y: numpy array of shape (n_examples)
        n_rows: numpy of rows to print
    """
    for example_num in range(n_rows):
        for feature in X[example_num]:
            print("{:.4f}".format(feature), end=",")

        if example_num == len(X)-1:
            print(y[example_num],end="")
        else:
            print(y[example_num])
# array of values without y (class)
x = processed_df.drop(columns=class_col).values
# array of y values
y = processed_df[class_col].values

print_data(x, y)

0.4628,0.5406,0.5113,0.4803,0.7380,0.4699,0.1196,1
0.4900,0.5547,0.5266,0.5018,0.7319,0.4926,0.8030,1
0.6109,0.6847,0.6707,0.5409,0.8032,0.6253,0.1185,0
0.6466,0.6930,0.6677,0.5961,0.7601,0.6467,0.2669,0
0.6712,0.6233,0.4755,0.8293,0.3721,0.6803,0.4211,1
0.2634,0.2932,0.2414,0.4127,0.5521,0.2752,0.2825,1
0.8175,0.9501,0.9515,0.5925,0.9245,0.8162,0.0000,0
0.3174,0.3588,0.3601,0.3908,0.6921,0.3261,0.8510,1
0.3130,0.3050,0.2150,0.5189,0.3974,0.3159,0.4570,1
0.5120,0.5237,0.4409,0.6235,0.5460,0.5111,0.3155,1


## **2. Build Classifiers**

- Part 1:  Logistic Regression, Naïve Bayes
- Part 2:  KNN, Decision Tree, Ada Boost, Gradient Boost, Random Forest, SVM

### Part 1: Cross-validation without parameter tuning

In [31]:
## Setting the 10 fold stratified cross-validation
cvKFold=StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

# The stratified folds from cvKFold should be provided to the classifiers

In [32]:
# Logistic Regression
logreg = LogisticRegression(random_state=0)
logreg_scores = cross_val_score(logreg, x, y, cv=cvKFold)
logreg_acc = logreg_scores.mean()

In [33]:
# Naïve Bayes
nb = GaussianNB()
nb_scores = cross_val_score(nb, x, y, cv=cvKFold)
nb_acc = nb_scores.mean()

### Part 1 Results


In [34]:
# Print results for each classifier in part 1 to 4 decimal places here:
print("LogR average cross-validation accuracy: {:.4f}".format(logreg_acc))
print("NB average cross-validation accuracy: {:.4f}".format(nb_acc))

LogR average cross-validation accuracy: 
NB average cross-validation accuracy: 


### Part 2: Cross-validation with parameter tuning

In [35]:
# KNN 
# parameters you may consider
k = [1, 3, 5, 7]
p = [1, 2]


In [36]:
# Decision Tree 
# parameters you may consider
max_depth = [3, 5, 7, 10]
min_samples_split = [2, 5, 10]
min_samples_leaf = [1, 2, 4]

In [37]:
# Ada Boost
# parameters you may consider
n_estimators = [50, 100, 150]
learning_rate = [0.1, 0.2, 0.3, 0.5]

In [38]:
# Gradient Boost
# parameters you may consider
max_depth = [1, 3, 5, 7]
n_estimators = [50, 100, 150]
learning_rate = [0.1, 0.2, 0.3, 0.5]

In [39]:
# Random Forest
# You should use RandomForestClassifier from sklearn.ensemble with information gain and max_features set to ‘sqrt’.
# parameters you may consider
n_estimators = [10, 30, 60, 100]
max_leaf_nodes = [6, 12]



In [40]:
# SVM
# parameters you may consider
C = [0.01, 0.1, 1, 5]
gamma = [0.01, 0.1, 1, 10]
# optional
kernel = []


### Part 2: Results

In [41]:
# Perform Grid Search with 10-fold stratified cross-validation (GridSearchCV in sklearn). 
# The stratified folds from cvKFold should be provided to GridSearchV

# This should include using train_test_split from sklearn.model_selection with stratification and random_state=0
# Print results for each classifier here. All the reported results should be printed to 4 decimal places except for the integers such as "k", "p", n_estimators" and "max_leaf_nodes".

# example printing:
print("KNN best k: ")
print("KNN best p: ")
print("KNN cross-validation accuracy: ")
print("KNN test set accuracy: ")

...

print("RF best n_estimators: ")
print("RF best max_leaf_nodes: ")
print("RF cross-validation accuracy: ")
print("RF test set accuracy: ")
print("RF test set macro average F1: ")
print("RF test set weighted average F1: ")

KNN best k: 
KNN best p: 
KNN cross-validation accuracy: 
KNN test set accuracy: 
RF best n_estimators: 
RF best max_leaf_nodes: 
RF cross-validation accuracy: 
RF test set accuracy: 
RF test set macro average F1: 
RF test set weighted average F1: 


### Test your code

In [42]:
#load the test dataset to test out your model 


## **3. Reflection and Discussion**



## **AI Acknowledgement**